In [ ]:
!poetry run jupyter lab --ServerApp.token='' --ServerApp.password=''


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
repo_root = pathlib.Path().resolve()  # adjust if you launched Jupyter in a subdir
# sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))



In [ ]:
# 1) Hot-reload while you tweak code
%load_ext autoreload
%autoreload 2



In [ ]:

from plainera_unacronym.nlp.execute import run_detection

# 2) Sanity test (text path)
text = """We met at 10 AM in the NHS ward. IT (Information Technology) stands for the tech team.
OK, let's proceed. No problem if R&D needs time."""
json_str = run_detection(text, parallel=False, caps_ratio=0.7, pretty=True, as_json=True)
print(json_str)   # pretty JSON in the cell


In [ ]:
# tests/test_acronyms_rd.py
from plainera_unacronym.nlp.execute import run_detection
import json


text = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads."
data = json.loads(run_detection(text, as_json=True))
keys = set(data["unique_acronyms"].keys())
print(keys)
assert "R&D" in keys
assert data["unique_acronyms"]["R&D"]["confidence"] >= 0.60


In [ ]:
# --- setup (adjust path if needed) ---
import sys, pathlib, json

try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

from plainera_unacronym.nlp.execute import run_detection


def detect(text: str, **kwargs) -> dict:
    """Call your detector and return a parsed dict."""
    return json.loads(run_detection(text, as_json=True, **kwargs))

In [ ]:



# --- tiny test harness ---
PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1

def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def counts_by_acronym(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out


# --- tests ---

def test_rd_detected_and_normalized():
    d = detect("We’ll loop in R & D after the NHS workshop.")
    assert "R&D" in keys(d), f"got {keys(d)}"
    assert d["unique_acronyms"]["R&D"]["confidence"] >= 0.60

def test_ok_and_am_drop_in_normal_prose():
    d = detect("OK, let's meet at 10:30 AM after lunch.")
    ks = keys(d)
    assert "OK" not in ks, f"OK leaked in: {ks}"
    assert "AM" not in ks, f"AM leaked in: {ks}"

def test_it_drops_as_pronoun_but_kept_with_definition():
    d1 = detect("IT was raining when we arrived.")
    assert "IT" not in keys(d1), f"pronoun IT should drop, got {keys(d1)}"

    d2 = detect("IT (Information Technology) owns the LDAP stack.")
    assert "IT" in keys(d2), f"IT with definition should keep, got {keys(d2)}"
    assert d2["unique_acronyms"]["IT"]["confidence"] >= 0.72

def test_stands_for_directional_rightward():
    d = detect("AM stands for amplitude modulation. OK, fine.")
    assert "AM" in keys(d), f"AM in definitional context should keep, got {keys(d)}"
    assert d["unique_acronyms"]["AM"]["confidence"] >= 0.72
    assert "OK" not in keys(d), f"OK should drop, got {keys(d)}"

def test_curly_apostrophe_and_hyphen_variants():
    d = detect("O’RAN and USB-C are on the agenda with the NHS.")
    ks = keys(d)
    assert "O’RAN" in ks, f"O’RAN missing, got {ks}"
    assert "USB-C" in ks, f"USB-C missing, got {ks}"
    assert "NHS" in ks

def test_mixed_alnum_kept():
    d = detect("H2O and MP3 appear in the doc.")
    ks = keys(d)
    assert "H2O" in ks, f"H2O missing, got {ks}"
    assert "MP3" in ks, f"MP3 missing, got {ks}"

def test_length_aware_threshold_blocks_bare_two_letter():
    d = detect("We will use AI and GPU for this.")
    ks = keys(d)
    assert "AI" not in ks, f"AI should drop under 2-letter threshold, got {ks}"
    assert "GPU" in ks, f"GPU should pass, got {ks}"

def test_company_suffixes_drop_without_definition():
    d = detect("Acme LTD and Example PLC signed the MOU.")
    ks = keys(d)
    assert "LTD" not in ks, f"LTD leaked in: {ks}"
    assert "PLC" not in ks, f"PLC leaked in: {ks}"
    # don't assert on MOU (might or might not be detected depending on your config)

def test_parallel_matches_serial_results():
    base = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
    big = base * 200  # large enough to trigger parallel path (if enabled)
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel), f"unique sets differ: {keys(serial)} vs {keys(parallel)}"
    assert counts_by_acronym(serial) == counts_by_acronym(parallel), \
        f"occurrence counts differ: {counts_by_acronym(serial)} vs {counts_by_acronym(parallel)}"


# --- run all ---
tests = [
    ("R&D detected & normalized", test_rd_detected_and_normalized),
    ("OK/AM drop in prose", test_ok_and_am_drop_in_normal_prose),
    ("IT: pronoun drops, definition keeps", test_it_drops_as_pronoun_but_kept_with_definition),
    ("'stands for' directional", test_stands_for_directional_rightward),
    ("Curly apostrophe & hyphen", test_curly_apostrophe_and_hyphen_variants),
    ("Mixed alnum kept", test_mixed_alnum_kept),
    ("Length-aware threshold on 2-letter", test_length_aware_threshold_blocks_bare_two_letter),
    ("Company suffixes drop", test_company_suffixes_drop_without_definition),
    ("Parallel parity", test_parallel_matches_serial_results),
]

for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
# --- longer notebook tests (no pytest needed) ---
import sys, pathlib, json, random, textwrap, itertools as it

# ensure package importable in notebook
try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

from plainera_unacronym.nlp.execute import run_detection


def detect(text: str, **kwargs) -> dict:
    return json.loads(run_detection(text, as_json=True, **kwargs))


PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1

def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def count_by_key(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out


# 1) Long paragraph with multiple signals (defs, separators, mixed alnum, distractors)
def test_long_mixed_paragraph():
    text = textwrap.dedent("""
        At 10:30 AM we met the NHS analytics team. IT (Information Technology) owns LDAP and SSO.
        The R & D unit collaborates with the GPU cluster for H2O simulations and MP3 decoding benchmarks.
        OK, let's add O’RAN to the agenda alongside USB-C adapters. Later, AM stands for amplitude modulation in RF.
        We will avoid dotted initialisms like U.S. here and noisy tokens like C++ or A+B which are not acronyms.
    """).strip()
    d = detect(text)
    ks = keys(d)
    # keep
    assert "NHS" in ks
    assert "IT" in ks and d["unique_acronyms"]["IT"]["confidence"] >= 0.72
    assert "R&D" in ks  # normalized from "R & D"
    assert "GPU" in ks and "H2O" in ks and "MP3" in ks
    assert "O’RAN" in ks and "USB-C" in ks
    # drop
    assert "OK" not in ks           # interjection
    assert "AM" in ks               # kept because of definitional sentence later ("AM stands for ...")
    assert "US" not in ks           # dotted variant shouldn't be matched by your pattern
    # sanity: C++ and A+B not treated as acronyms
    assert all(acr not in {"C++", "A+B"} for acr in ks)

# 2) Very long distance between token and definition should NOT boost (directional + windowed)
def test_directional_window_limit():
    filler = " lorem ipsum dolor sit amet," * 20  # ~600+ chars
    text = f"IT {filler} stands for Information Technology."  # 'stands for' too far to the right
    d = detect(text)
    assert "IT" not in keys(d), "IT should not be boosted by far-away 'stands for'"

# 3) First-occurrence mapping remains stable with normalization
def test_first_occurrence_normalization_stable():
    text = "R & D met R&D after lunch. R & D then emailed."
    d = detect(text)
    assert "R&D" in keys(d)
    first = d["unique_acronyms"]["R&D"]
    # The first span should be the earliest occurrence in text
    assert first["start_offset"] == text.index("R & D")

# 4) Repetition & counts in a larger synthetic corpus
def test_large_repetition_counts():
    base = "NHS and R&D work with GPU and USB-C. "
    text = base * 250  # 250 repetitions
    d = detect(text)
    counts = count_by_key(d)
    # Expect ~250 occurrences each (some tokens might appear twice per base, adjust as needed)
    for acr in ["NHS", "R&D", "GPU", "USB-C"]:
        assert acr in counts and counts[acr] >= 230, f"{acr} count too low: {counts.get(acr)}"

# 5) Hyphen/en-dash noise: GPU should still be detected; right-hand lower token shouldn't be needed
def test_gpu_with_following_dash_word():
    text = "We tested GPU–accelerated pipelines and GPU-accelerated kernels yesterday."
    d = detect(text)
    ks = keys(d)
    assert "GPU" in ks, "GPU should be detected even when followed by dash-word"
    # Ensure we didn't create a weird token spanning the dash
    assert all(o["acronym"] != "GPU–accelerated" for o in d["occurrences"])

# 6) Company suffixes and short common uppers drop (unless defined)
def test_company_suffixes_drop_and_ok_drops():
    text = "Acme LTD acquired Example PLC. OK, moving on. DR Smith arrived at 7 AM."
    d = detect(text)
    ks = keys(d)
    assert "LTD" not in ks and "PLC" not in ks
    assert "OK" not in ks
    assert "AM" not in ks  # time-of-day
    # DR is in non_acronym_upper → drop unless explicit definition
    assert "DR" not in ks

# 7) Parenthetical definition rescues otherwise noisy tokens
def test_parenthetical_rescue_for_ok_and_short_tokens():
    text = "OK (Object Kernel) appeared in legacy docs; AI (Artificial Intelligence) and IT (Information Technology) led."
    d = detect(text)
    ks = keys(d)
    assert "OK" in ks and d["unique_acronyms"]["OK"]["confidence"] >= 0.72
    assert "AI" in ks and d["unique_acronyms"]["AI"]["confidence"] >= 0.72
    assert "IT" in ks

# 8) Parallel == serial parity on a long doc
def test_parallel_parity_long_doc():
    para = (
        "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
        "O’RAN and USB-C were discussed. GPU outperformed CPU. "
    )
    big = para * 300
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel)
    assert count_by_key(serial) == count_by_key(parallel)

# 9) Context window snaps to sentence boundaries (lenient check)
def test_context_window_sentence_bounds():
    text = "Alpha. We met the NHS team today and they agreed. Beta."
    d = detect(text)
    nhs_occ = next(o for o in d["occurrences"] if o["acronym"] == "NHS")
    left, right = nhs_occ["context_window"]
    segment = text[left:right]
    assert segment.strip().endswith("agreed."), f"window right seems off: {segment!r}"
    assert segment.strip().startswith("We met the"), f"window left seems off: {segment!r}"

# 10) Long definitional form lines (parentheses) still boost within limit
def test_long_parenthetical_still_boosts_within_limit():
    long_def = "(Information Technology and related shared infrastructure services)"
    text = f"IT {long_def} owns the platform."
    d = detect(text)
    assert "IT" in keys(d) and d["unique_acronyms"]["IT"]["confidence"] >= 0.72


# --- run all ---
tests = [
    ("Long mixed paragraph", test_long_mixed_paragraph),
    ("Directional window limit", test_directional_window_limit),
    ("First-occurrence normalization", test_first_occurrence_normalization_stable),
    ("Large repetition counts", test_large_repetition_counts),
    ("GPU with following dash word", test_gpu_with_following_dash_word),
    ("Company suffixes & OK drop", test_company_suffixes_drop_and_ok_drops),
    ("Parenthetical rescue for short tokens", test_parenthetical_rescue_for_ok_and_short_tokens),
    ("Parallel parity on long doc", test_parallel_parity_long_doc),
    ("Context window sentence bounds", test_context_window_sentence_bounds),
    ("Long parenthetical boost", test_long_parenthetical_still_boosts_within_limit),
]

PASSED = FAILED = 0
for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")
